In [2]:
import pandas as pd
import sqlite3
import numpy as np


# Player_stats_field
conn = sqlite3.connect("kickbase.db")

query_field_players = "SELECT * FROM player_stats_field"
query_goalkeeper = "SELECT * FROM player_stats_gk"
query_team_stats = "SELECT * FROM team_stats"


df_field_players = pd.read_sql_query(query_field_players, conn)
df_goalkeeper = pd.read_sql_query(query_goalkeeper, conn)
df_team_stats = pd.read_sql_query(query_field_players, conn)
conn.close()

df_field_players = df_field_players.sort_values(by=['player_id', 'season', 'matchday']).reset_index(drop=True)   
df_goalkeeper = df_goalkeeper.sort_values(by=['player_id', 'season', 'matchday']).reset_index(drop=True)

In [3]:
def points_avg_and_trend(df):
    df["points_filled"] = df['points'].fillna(0)

    df["points_avg_last_3"] = (
    df.groupby(["player_id", "season"])["points_filled"] 
    .shift(1)
    .rolling(window=3)
    .mean()
)
    df["points_avg_last_5"] = (
    df.groupby(["player_id", "season"])["points_filled"]
    .shift(1)
    .rolling(window=5)
    .mean()
)
    df["points_trend"] = df["points_avg_last_3"] - df["points_avg_last_5"]
    df = df.drop(columns=["points_filled"])
    return df

In [4]:
df_field_players = points_avg_and_trend(df_field_players)
df_field_players[["player_id", "matchday", "points", "points_avg_last_3", "points_avg_last_5", "points_trend"]].head(20)

,player_id,matchday,points,points_avg_last_3,points_avg_last_5,points_trend
0,1,1,46.0,NaN,NaN,NaN
1,1,2,40.0,NaN,NaN,NaN
2,1,3,34.0,NaN,NaN,NaN
3,1,4,92.0,40.000000,NaN,NaN
4,1,5,99.0,55.333333,NaN,NaN
5,1,6,52.0,75.000000,62.2,12.800000
6,1,7,70.0,81.000000,63.4,17.600000
7,1,8,70.0,73.666667,69.4,4.266667
8,1,9,144.0,64.000000,76.6,-12.600000
9,1,10,31.0,94.666667,87.0,7.666667


In [5]:
def minutes_avg_and_trend(df):
    df["minutes_filled"] = df["minutes"].fillna(0)
    df["minutes_avg_last_3"] = (
    df.groupby(["player_id", "season"])["minutes_filled"]
    .shift(1)
    .rolling(window=3)
    .mean()
)
    df["minutes_avg_last_5"] = (
    df.groupby(["player_id", "season"])["minutes_filled"]
    .shift(1)
    .rolling(window=5)
    .mean()
)
    df.drop(columns=["minutes_filled"])
    return df

In [6]:
df_field_players = minutes_avg_and_trend(df_field_players)
df_field_players[["player_id", "matchday", "minutes", "minutes_avg_last_3", "minutes_avg_last_5"]].head(20)

,player_id,matchday,minutes,minutes_avg_last_3,minutes_avg_last_5
0,1,1,93,NaN,NaN
1,1,2,96,NaN,NaN
2,1,3,99,NaN,NaN
3,1,4,94,96.000000,NaN
4,1,5,99,96.333333,NaN
5,1,6,98,97.333333,96.2
6,1,7,100,97.000000,97.2
7,1,8,97,99.000000,98.0
8,1,9,95,98.333333,97.6
9,1,10,94,97.333333,97.8


In [7]:
def points_p90(df):
    df["points_p90_last_5"] = (
        (df["points_avg_last_5"] / df["minutes_avg_last_5"]) * 90
    )
    df["points_p90_last_3"] = (
    (df["points_avg_last_3"] / df["minutes_avg_last_3"]) * 90
)
    return df

In [8]:
df_field_players = points_p90(df_field_players)
df_field_players[["player_id", "matchday", "minutes_avg_last_3", "minutes_avg_last_5", "points_p90_last_5","points_p90_last_5"]].head(20)

,player_id,matchday,minutes_avg_last_3,minutes_avg_last_5,points_p90_last_5,points_p90_last_5
0,1,1,NaN,NaN,NaN,NaN
1,1,2,NaN,NaN,NaN,NaN
2,1,3,NaN,NaN,NaN,NaN
3,1,4,96.000000,NaN,NaN,NaN
4,1,5,96.333333,NaN,NaN,NaN
5,1,6,97.333333,96.2,58.191268,58.191268
6,1,7,97.000000,97.2,58.703704,58.703704
7,1,8,99.000000,98.0,63.734694,63.734694
8,1,9,98.333333,97.6,70.635246,70.635246
9,1,10,97.333333,97.8,80.061350,80.061350


In [9]:
def p90(df, efficiency_cols):

    for col in efficiency_cols:
        df[f"{col}_filled"] = df[col].fillna(0)

        df[f"{col}_avg_last_3"] = (
            df.groupby(["player_id", "season"])[f"{col}_filled"]
            .shift(1)
            .rolling(window=3)
            .mean()
        )

        df[f"{col}_p90_last_3"] = df[f"{col}_avg_last_3"] / df["minutes_avg_last_3"] * 90
        df[f"{col}_p90_last_3"] = df[f"{col}_p90_last_3"].replace([np.inf, -np.inf], np.nan).fillna(0)  
        
        df = df.drop(columns=[f"{col}_filled", f"{col}_avg_last_3"])
    
    return df


In [10]:
efficiency_cols = [
    "gewonnene_zweikaempfe",
    "schuesse_aufs_tor",
    "torschuss_vorlagen",
    "kreierte_grosschancen",
    "geklaerte_baelle",
    "abgefangene_baelle",
    "balleroberungen",
    "geblockte_baelle",
    "fehler_vor_gegentor",
    "begangene_fouls",
    "ballverluste"
]
df_field_players = p90(df_field_players, efficiency_cols)
df_field_players[["player_id", "matchday", "season", "schuesse_aufs_tor", "schuesse_aufs_tor_p90_last_3"]].head(20)

,player_id,matchday,season,schuesse_aufs_tor,schuesse_aufs_tor_p90_last_3
0,1,1,2024/2025,1,0.000000
1,1,2,2024/2025,0,0.000000
2,1,3,2024/2025,0,0.000000
3,1,4,2024/2025,0,0.312500
4,1,5,2024/2025,0,0.000000
5,1,6,2024/2025,0,0.000000
6,1,7,2024/2025,0,0.000000
7,1,8,2024/2025,0,0.000000
8,1,9,2024/2025,0,0.000000
9,1,10,2024/2025,0,0.000000


In [11]:
def ratios(df, ratio_pairs):
    for succes_col, total_col in ratio_pairs:
        df[f"{succes_col}_filled"] = df[succes_col].fillna(0)
        df[f"{total_col}_filled"] = df[total_col].fillna(0)

        df[f"{succes_col}_sum_last_3"] = (
        df.groupby(["player_id", "season"])[f"{succes_col}_filled"]        
        .shift(1)
        .rolling(window=3)
        .sum()
        )

        df[f"{total_col}_sum_last_3"] = (
        df.groupby(["player_id", "season"])[f"{total_col}_filled"]        
        .shift(1)
        .rolling(window=3)
        .sum()
        )

        df[f"{succes_col}_ratio_last_3"] = df[f"{succes_col}_sum_last_3"] / df[f"{total_col}_sum_last_3"]
        df[f"{succes_col}_ratio_last_3"] = df[f"{succes_col}_ratio_last_3"].replace([np.inf, -np.inf], np.nan).fillna(0)

        df = df.drop(columns=[
            f"{succes_col}_filled",
            f"{succes_col}_sum_last_3",
            f"{total_col}_filled",
            f"{total_col}_sum_last_3"
        ])

        return df

In [12]:
ratio_pairs = [
    ("erfolgreiche_paesse", "paesse_gesamt"), 
    ("gewonnene_zweikaempfe", "gewonnene_zweikaempfe_gesamt"),
    ("gewonnene_luftkaempfe", "gewonnene_luftkaempfe_gesamt"),
    ("erfolgreiche_tacklings", "tacklings_gesamt"),
    ("erfolgreiche_dribblings", "dribblings_gesamt"),
    ("schussgenauigkeit", "schussgenauigkeit_gesamt")
]
df_field_players = ratios(df_field_players, ratio_pairs)
df_field_players[["player_id", "matchday", "erfolgreiche_paesse_ratio_last_3", "gewonnene_zweikaempfe_ratio_last_3", "gewonnene_luftkaempfe_ratio_last_3"]].head(20)
df_field_players[["player_id", "matchday", "erfolgreiche_tacklings_ratio_last_3", "erfolgreiche_dribblings_ratio_last_3", "schussgenauigkeit_ratio_last_3"]].head(20)


KeyError: "['gewonnene_zweikaempfe_ratio_last_3', 'gewonnene_luftkaempfe_ratio_last_3'] not in index"

In [14]:
def form_trends(df, form_trend_cols):
    for col in form_trend_cols:
        df[f"{col}_filled"] = df[col].fillna(0)
        
        df[f"{col}_avg_last_3"] = (
            df.groupby(["player_id", "season"])[f"{col}_filled"]
            .shift(1).rolling(window=3).mean()
        )
        
        df[f"{col}_avg_last_5"] = (
            df.groupby(["player_id", "season"])[f"{col}_filled"]
            .shift(1).rolling(window=5).mean()
        )
        
        df[f"{col}_trend"] = df[f"{col}_avg_last_3"] - df[f"{col}_avg_last_5"]
        
        df = df.drop(columns=[f"{col}_filled"])

        return df

In [4]:
form_trend_cols = [
    "goals_own_team", 
    "goals_enemy_team", 
    "match_result", 
    "grade",
    "points_per_value"           
]

df_field_players = form_trends(df_field_players)

NameError: name 'form_trends' is not defined

In [16]:
df_field_players.head(20)

,id,player_id,season,matchday,date,points,minutes,points_per_minute,market_value,points_per_value,...,torschuss_vorlagen_p90_last_3,kreierte_grosschancen_p90_last_3,geklaerte_baelle_p90_last_3,abgefangene_baelle_p90_last_3,balleroberungen_p90_last_3,geblockte_baelle_p90_last_3,fehler_vor_gegentor_p90_last_3,begangene_fouls_p90_last_3,ballverluste_p90_last_3,erfolgreiche_paesse_ratio_last_3
0,32,1,2024/2025,1,2024-08-25T15:30:00Z,46.0,93,0.494624,None,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,33,1,2024/2025,2,2024-08-30T18:30:00Z,40.0,96,0.416667,None,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,34,1,2024/2025,3,2024-09-15T13:30:00Z,34.0,99,0.343434,None,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,35,1,2024/2025,4,2024-09-22T17:30:00Z,92.0,94,0.978723,None,NaN,...,0.000000,0.000000,3.437500,0.937500,0.625000,0.312500,0.000000,0.000000,0.312500,0.838565
4,36,1,2024/2025,5,2024-09-28T13:30:00Z,99.0,99,1.000000,None,NaN,...,0.311419,0.311419,3.737024,1.245675,0.311419,0.311419,0.000000,0.311419,0.622837,0.844828
5,37,1,2024/2025,6,2024-10-05T16:30:00Z,52.0,98,0.530612,None,NaN,...,0.308219,0.308219,3.390411,0.616438,0.308219,0.616438,0.000000,0.308219,0.308219,0.761905
6,38,1,2024/2025,7,2024-10-18T18:30:00Z,70.0,100,0.700000,None,NaN,...,0.618557,0.309278,3.092784,0.618557,0.309278,0.309278,0.000000,0.309278,0.618557,0.786885
7,39,1,2024/2025,8,2024-10-26T13:30:00Z,70.0,97,0.721649,None,NaN,...,0.303030,0.000000,2.424242,0.000000,1.515152,0.606061,0.000000,0.000000,0.303030,0.836066
8,40,1,2024/2025,9,2024-11-02T14:30:00Z,144.0,95,1.515789,None,NaN,...,0.305085,0.000000,3.050847,0.000000,1.525424,0.305085,0.305085,0.000000,0.305085,0.878613
9,41,1,2024/2025,10,2024-11-09T14:30:00Z,31.0,94,0.329787,None,NaN,...,0.000000,0.000000,3.698630,0.308219,1.849315,1.232877,0.308219,0.000000,0.000000,0.902439


In [3]:
def sums_cards_goals_assists(df):
    df["cards_total"] = df["yellow_cards"].fillna(0) + df["yellow_red_cards"].fillna(0) * 2 + df["red_cards"].fillna(0) * 3

    sum_cols = ["goals", "assists", "cards_total"]

    for col in sum_cols:
        df[f"{col}_sum_last_5"] = (
            df.groupby(["player_id", "season"])[col]
            .shift(1).rolling(window=5).sum().fillna(0)
        )
        
    df = df.drop(columns=["cards_total"])

    return df